## 6. Assignment: Audio Transcription Using Full and Quantized Open-Source Models

### Objective
Build an audio transcription pipeline using **open-source Hugging Face speech models** and compare the performance of a **full model** and a **quantized model**.

### Task
1. Choose an open-source audio transcription model from Hugging Face.
2. Use one **full-precision model** and one **quantized version** of the model, or quantize the model yourself if needed.
3. Transcribe the same set of audio samples using both models.
4. Compare the transcription outputs and analyze the errors.
5. Report the quality and efficiency differences between the two models.

In [1]:
!pip install jiwer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 33.4 MB/s eta 0:00:00


In [2]:
import torch
import librosa
import time
import os
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from jiwer import wer, cer

In [ ]:
original_audio_file_path = "/audios/OAF_peg_fear"
noise_audio_file_path = "/audios/RDR.wav"

if os.path.exists(noise_audio_file_path) and os.path.exists(original_audio_file_path):
    print("File exists.")
else:
    print("File does not exist.")
    exit(1)
waveform, sample_rate = librosa.load(noise_audio_file_path, sr=16000)

print("Sample Rate:", sample_rate)
print("Duration:", len(waveform) / sample_rate)

File exists.
Sample Rate: 16000
Duration: 5.0033125


In [ ]:
reference_text = "take a gamble that love exists and do a loving act"
reference_text_2 = "say the word pig"

In [17]:
# note: never forget to set HF_TOKEN in env var
asr_processor = WhisperProcessor.from_pretrained("openai/whisper-tiny")
asr_model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-tiny")
asr_model.eval()
model_inputs = asr_processor(waveform, sampling_rate=16000, return_tensors="pt")

Loading weights:   0%|          | 0/167 [00:00<?, ?it/s]

In [18]:
start_time = time.time()

generated_ids_full = asr_model.generate(model_inputs.input_features)

full_model_inference_time = time.time() - start_time
full_model_transcript = asr_processor.batch_decode(generated_ids_full, skip_special_tokens=True)[0]

print("Full Model Transcription:\n")
print(full_model_transcript)
print("\nInference Time:", full_model_inference_time)

Full Model Transcription:

 Take a gamble of a love existence and do we love in it?

Inference Time: 1.7505722045898438


In [19]:
quantized_asr_model = torch.quantization.quantize_dynamic(asr_model, {torch.nn.Linear}, dtype=torch.qint8)
quantized_asr_model.eval()
print("Quantization Completed!")

/tmp/ipykernel_1642/3525308928.py:1: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_asr_model = torch.quantization.quantize_dynamic(asr_model, {torch.nn.Linear}, dtype=torch.qint8)


Quantization Completed!


In [20]:
start_time = time.time()

generated_ids_quant = quantized_asr_model.generate(model_inputs.input_features)
quantized_model_inference_time = time.time() - start_time
quantized_model_transcript = asr_processor.batch_decode(generated_ids_quant, skip_special_tokens=True)[0]

print("Quantized Model Transcription:\n")
print(quantized_model_transcript)
print("\nInference Time:", quantized_model_inference_time)

Quantized Model Transcription:

 I'm doing nothing.

Inference Time: 0.9918432235717773


In [21]:
torch.save(asr_model.state_dict(), "whisper_full.pth")
torch.save(quantized_asr_model.state_dict(), "whisper_quantized.pth")

full_model_size_mb = os.path.getsize("whisper_full.pth") / (1024 * 1024)
quantized_model_size_mb = os.path.getsize("whisper_quantized.pth") / (1024 * 1024)

print(f"Full Model Size : {full_model_size_mb:.2f} MB")
print(f"Quantized Model Size : {quantized_model_size_mb:.2f} MB")

Full Model Size : 144.10 MB
Quantized Model Size : 115.90 MB


In [24]:
word_error_rate_1 = wer(reference_text, full_model_transcript)
char_error_rate_1 = cer(reference_text, full_model_transcript)


word_error_rate_2 = wer(reference_text, quantized_model_transcript)
char_error_rate_2 = cer(reference_text, quantized_model_transcript)

print("For Full Model")
print("WER :", word_error_rate_1)
print("CER :", char_error_rate_1)

print("For Quantized Model")
print("WER :", word_error_rate_2)
print("CER :", char_error_rate_2)

For Full Model
WER : 0.7272727272727273
CER : 0.34
For Quantized Model
WER : 1.0
CER : 0.8


In [26]:
import pandas as pd

comparison_df = pd.DataFrame({
    "Metric": ["Inference Time", "Model Size", "WER", "CER"],
    "Full Model": [round(full_model_inference_time, 2), round(full_model_size_mb, 2), round(word_error_rate_1, 4), round(char_error_rate_1, 4)],
    "Quantized Model": [round(quantized_model_inference_time, 2), round(quantized_model_size_mb, 2), round(word_error_rate_2, 4), round(char_error_rate_2, 4)]
})

comparison_df

,Metric,Full Model,Quantized Model
0,Inference Time,1.7500,0.99
1,Model Size,144.1000,115.90
2,WER,0.7273,1.00
3,CER,0.3400,0.80


# Observations

The full-precision Whisper Tiny model produced the most accurate transcription of the speech sample.

The dynamically quantized model generated a transcription with only a few minor changes if the audio is without noise if there is noise then the output is completely distorted.

The quantized model was approximately 40% faster than the full model and reduced storage requirements by about 20%, demonstrating that dynamic quantization improves computational efficiency with reduction in transcription accuracy.